# Perception: point at the ball

The robot's head camera sees a red ball. This module finds the ball in the
picture, works out where it is in front of the robot, and points the right arm
at it. Then it closes the loop: look again, correct, and repeat until the
answer stops changing.

It builds on [Joints and frames](joints_and_frames.ipynb) (where the camera
is) and [Inverse kinematics](inverse_kinematics.ipynb) (how to reach).

Run on the mock first. Its camera draws what the robot's head camera would
see, including the ball.

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME, camera="render") if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. Look

`r.camera.read()` gives one picture: an array of height × width × 3
colours (red, green, blue, 0–255).

In [ ]:
t, img = r.camera.read()
print("picture", img.shape, img.dtype)
plt.figure(figsize=(6, 4.5)); plt.imshow(img); plt.title("what the robot sees"); plt.show()

## 2. Find the ball

A pixel is "ball" when it is clearly red: much more red than green or
blue. The ball's position in the picture is the centre (mean *u*, mean *v*)
of those pixels; its size is how many there are. *u* counts columns from
the left, *v* rows from the top.

`find_ball(img)` returns `(u, v, area, width, height)`, or `None` when fewer
than 30 pixels are red. The width and height matter in part 5.

In [ ]:
def find_ball(img, min_area=30):
    rgb = img.astype(int)
    red = (rgb[..., 0] > 100) & (rgb[..., 0] > 2 * rgb[..., 1]) & (rgb[..., 0] > 2 * rgb[..., 2])
    v, u = np.nonzero(red)
    if len(u) < min_area:
        return None
    return float(u.mean()), float(v.mean()), int(len(u)), int(np.ptp(u)) + 1, int(np.ptp(v)) + 1

In [ ]:
ball = find_ball(img)
print("ball at pixel (%.1f, %.1f), %d pixels, %d x %d" % ball)
plt.figure(figsize=(6, 4.5)); plt.imshow(img)
plt.plot(ball[0], ball[1], "c+", ms=20, mew=2); plt.title("found it"); plt.show()

## 3. From a pixel to a direction

The camera is a pinhole: a point at (*x* forward, *y* left, *z* up) in the
camera's frame lands on pixel

  *u* = *c<sub>x</sub>* − *f* · *y* / *x*,  *v* = *c<sub>y</sub>* − *f* · *z* / *x*

with *f* the focal length in pixels and (*c<sub>x</sub>*, *c<sub>y</sub>*)
the picture's centre. Turned around, pixel (*u*, *v*) is the direction
(1, −(*u* − *c<sub>x</sub>*)/*f*, −(*v* − *c<sub>y</sub>*)/*f*).

The camera is bolted to the head (`head_link`). These are its mount and
lens; the mock and the simulator use exactly these, the robot's own
calibration is within a few pixels.

In [ ]:
import xml.etree.ElementTree as ET

FX = FY = 352.1                                   # focal length (pixels)
CX, CY = 325.3, 234.6                             # where the lens centre lands (640 x 480)
MOUNT_XYZ = (0.07407, 0.06494, 0.56493)           # head_link -> camera (m)
MOUNT_RPY = (-0.01794, 0.81543, -0.02518)         # (rad): it looks ~47 deg down
BALL_RADIUS = 0.04                                # m

# Forward kinematics, from Joints and frames.
URDF = os.path.join(os.path.dirname(g1lab.__file__), "description", "g1_29dof_dx3_ros.urdf")
JOINT_OF_CHILD = {j.find("child").get("link"): j for j in ET.parse(URDF).getroot().findall("joint")}

def rot_axis(axis, q):
    k = np.asarray(axis, float) / np.linalg.norm(axis)
    K = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + math.sin(q) * K + (1 - math.cos(q)) * K @ K

def rot_rpy(roll, pitch, yaw):
    return rot_axis([0, 0, 1], yaw) @ rot_axis([0, 1, 0], pitch) @ rot_axis([1, 0, 0], roll)

def chain(link):
    out = []
    while link in JOINT_OF_CHILD:
        out.append(JOINT_OF_CHILD[link]); link = out[-1].find("parent").get("link")
    return out[::-1]

def fk(angles, link):
    T = np.eye(4)
    for j in chain(link):
        o, step = j.find("origin"), np.eye(4)
        step[:3, :3] = rot_rpy(*map(float, o.get("rpy", "0 0 0").split()))
        step[:3, 3] = list(map(float, o.get("xyz", "0 0 0").split()))
        T = T @ step
        if j.get("type") == "revolute":
            turn = np.eye(4)
            turn[:3, :3] = rot_axis(list(map(float, j.find("axis").get("xyz").split())),
                                    angles.get(j.get("name"), 0.0))
            T = T @ turn
    return T

def urdf_angles(js):
    a = {f"waist_{n}_joint": js[f"waist_{n}"] for n in ("yaw", "roll", "pitch")}
    for side in ("left", "right"):
        for name, q in zip(g1lab.ARM, js.arm(side)):
            a[f"{side}_{name}_joint"] = float(q)
    return a

MOUNT = np.eye(4)
MOUNT[:3, :3], MOUNT[:3, 3] = rot_rpy(*MOUNT_RPY), MOUNT_XYZ

def camera_pose(js):
    """The camera's 4x4 pose in the pelvis frame (x forward, y left, z up)."""
    return fk(urdf_angles(js), "head_link") @ MOUNT

T_cam = camera_pose(r.state.joints())
print("camera in the pelvis frame (m):", np.round(T_cam[:3, 3], 3))

`ball_position(ball, T_cam)` gives the ball's centre in the pelvis frame:

1. The direction to the ball in the camera frame (above), as a unit vector.
2. How far: the ball's radius on the picture is *r* = √(area / π) pixels,
   and it covers an angle α with tan α = *r* / *f*. A sphere of radius *R*
   seen under α is at distance *R* / sin α.
3. The point in the camera frame, then through `T_cam` to the pelvis frame.

In [ ]:
def ball_position(ball, T_cam):
    u, v, area = ball[:3]
    ray = np.array([1.0, -(u - CX) / FX, -(v - CY) / FY])
    ray /= np.linalg.norm(ray)
    alpha = math.atan(math.sqrt(area / math.pi) / FX)
    p_cam = ray * BALL_RADIUS / math.sin(alpha)
    return (T_cam @ np.append(p_cam, 1.0))[:3]

In [ ]:
B = ball_position(ball, T_cam)
print("the ball is at", np.round(B, 3), "m in the pelvis frame")
if HOST == "mock":                                # the mock knows where it put the ball
    truth = np.array([0.42, -0.12, 0.92 - 0.79])  # placed in the world; the pelvis stands 0.79 m up
    print(f"  the mock placed it at {truth}: off by {1000 * np.linalg.norm(B - truth):.0f} mm")

The distance along the camera's line of sight is the least certain. The ball is about 50 pixels across, so one pixel of error in its radius moves the distance by about 1 in, while one pixel of error in its centre moves it sideways by less than 0.1 in.

## 4. Point

"Pointing" here means putting the palm on the line from the right shoulder
to the ball, 0.25 m from the shoulder, or 8 cm short of the ball if that is
closer. The solver from [Inverse kinematics](inverse_kinematics.ipynb) finds
the angles.

In [ ]:
RIGHT = [f"right_{n}_joint" for n in g1lab.ARM]
home = info.home["right_arm"].copy()
lo, hi = info.limits["right_arm"].T

def palm(q7):
    return fk(dict(zip(RIGHT, q7)), "right_hand_palm_link")[:3, 3]

def jacobian(q7, eps=1e-6):
    p0, J = palm(q7), np.zeros((3, 7))
    for i in range(7):
        dq = np.array(q7, float); dq[i] += eps
        J[:, i] = (palm(dq) - p0) / eps
    return J

def ik(target, q_start, damping=0.05, iters=300, tol=1e-3, pull=0.05, max_step=0.05):
    """The damped-least-squares solver from Inverse kinematics."""
    q = np.array(q_start, float)
    for _ in range(iters):
        err = np.asarray(target) - palm(q)
        if np.linalg.norm(err) < tol:
            break
        err = err * min(1.0, max_step / np.linalg.norm(err))
        J = jacobian(q)
        Jinv = J.T @ np.linalg.inv(J @ J.T + damping**2 * np.eye(3))
        q = np.clip(q + Jinv @ err + (np.eye(7) - Jinv @ J) @ (pull * (home - q)), lo, hi)
    return q, float(np.linalg.norm(np.asarray(target) - palm(q)))

SHOULDER = fk({}, "right_shoulder_pitch_link")[:3, 3]    # it does not move with the arm
REACH = 0.25

`aim(B)` gives the palm target on the shoulder–ball line, and the arm angles
that reach it, starting the solver from `home`.

In [ ]:
def aim(B):
    d = np.asarray(B) - SHOULDER
    dist = np.linalg.norm(d)
    target = SHOULDER + min(REACH, dist - 0.08) * d / dist
    q, miss = ik(target, home)
    return target, q, miss

In [ ]:
def pointing_error(q7, B):
    """Angle (deg) between shoulder->palm and shoulder->ball."""
    a, c = palm(q7) - SHOULDER, np.asarray(B) - SHOULDER
    return math.degrees(math.acos(np.clip(a @ c / np.linalg.norm(a) / np.linalg.norm(c), -1, 1)))

target, q, miss = aim(B)
print(f"palm target {np.round(target, 3)}, solver miss {1000 * miss:.1f} mm, "
      f"pointing off by {pointing_error(q, B):.1f} deg")

## 5. Close the loop

One look is not enough: the estimate from part 3 has errors, and the robot
sways a little (in the simulator and on the robot), so the camera moves.
So: look, estimate, point, and look again, until the estimate moves by
less than 1 cm between two looks. Each look uses the camera's pose *at
that moment*.

There is a catch. Once the arm is up, **the hand can hide part of the
ball**. A partly hidden ball has fewer red pixels, so it looks smaller, so
it seems further away, so the arm moves, so a different part is hidden...
Try it: set `TRUST_ALL = True` below and watch the estimate jump about.

The fix: only trust a look when the ball looks whole. A whole ball is a
round patch that fills its own box: width ≈ height, and area ≈ π/4 ·
width · height. When it does not, keep the last estimate. When the hand
hides the ball look after look, it is pointing right at it (from the
head, a hand pointing at a ball covers it): stop.

In [ ]:
TRUST_ALL = False

def looks_whole(found, tol=0.12):
    u, v, area, w, h = found
    return abs(w - h) <= tol * max(w, h) and area >= (1 - tol) * math.pi / 4 * w * h

def look():
    """(estimate or None, picture, blob or None, whole?): one look, the camera where it is now."""
    js = r.state.joints()                         # the camera's pose now ...
    _, img = r.camera.read()                      # ... and what it sees now
    found = find_ball(img)
    if found is None:
        return None, img, None, False
    return ball_position(found, camera_pose(js)), img, found, TRUST_ALL or looks_whole(found)

with r.control() as c:
    B_est, hidden = None, 0
    for k in range(10):
        B_new, img, found, whole = look()
        if not whole:
            hidden += 1
            print(f"look {k}: the ball is " + ("partly hidden" if found else "not in view")
                  + ": keeping the last estimate")
            if B_est is not None and hidden >= 3:
                print("  the hand is in front of the ball: pointing at it")
                break
            continue
        hidden = 0
        moved = None if B_est is None else np.linalg.norm(B_new - B_est)
        B_est = B_new
        print(f"look {k}: ball at {np.round(B_est, 3)}"
              + ("" if moved is None else f", moved {1000 * moved:.0f} mm since the last look"))
        if moved is not None and moved < 0.01:
            break
        target, q, miss = aim(B_est)
        try:
            c.arms.move_to(right=q, duration=1.5)
        except g1lab.Refused as e:
            print("  refused:", e.reason)
            break
        time.sleep(0.5)                           # let the arm settle (the arm lags ~0.1 s)
        print(f"  pointing off by {pointing_error(r.state.joints().arm('right'), B_est):.1f} deg")
    time.sleep(1.0)
    c.arms.home()

plt.figure(figsize=(6, 4.5)); plt.imshow(img)
if found is not None:
    plt.plot(found[0], found[1], "c+", ms=20, mew=2)
plt.title("the last look"); plt.show()

**Try this.** Run part 5 again with `TRUST_ALL = True`. The estimate moves
further away rather than closer: a hidden ball has fewer red pixels, so it looks
smaller, and a smaller ball seems further off.

The pointing error above is measured with your own model (the joints you read,
through `fk()`). To measure it independently of the model, find the hand in the
picture: the arm is in view.

**Try this (on the mock).** Restart the mock with the ball somewhere else, e.g.
`g1lab.mock(name=NAME, camera="render", ball=(0.40, -0.30, 1.05))`, and run
parts 2 to 5 again. Look for a place where the ball is in view but the arm
cannot point at it, and see what the program does.